In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
df = pd.read_csv('AirQualityUCI.csv', sep=',', encoding='utf-8-sig')

# Drop unnamed or empty columns
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# a. Data Cleaning
df.replace(-200, np.nan, inplace=True)  # Replace -200 with NaN
df.dropna(subset=['CO(GT)'], inplace=True)  # Drop rows where target is NaN
df.fillna(df.mean(numeric_only=True), inplace=True)  # Fill other NaNs

# Convert date and time
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
df.drop(['Date', 'Time'], axis=1, inplace=True)

# b. Integration: set datetime index
df.set_index('Datetime', inplace=True)

# c. Transformation: normalize
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
scaled_columns = df.columns
df[scaled_columns] = scaler.fit_transform(df[scaled_columns])

# d. Modeling
X = df.drop('CO(GT)', axis=1)
y = df['CO(GT)']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# e. Output
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R² Score: {r2:.4f}\n")

# Clip predictions to [0, 1]
y_pred_corrected = np.clip(y_pred, 0, 1)

# Compare predictions vs actual
results_df = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred_corrected
})
print("Sample Predictions (first 10):")
print(results_df.head(10), "\n")

# Coefficients
coefficients = pd.Series(model.coef_, index=X.columns)
print("Feature Importances (Linear Regression Coefficients):")
print(coefficients.sort_values(ascending=False))


Mean Squared Error (MSE): 0.0016
R² Score: 0.8908

Sample Predictions (first 10):
     Actual  Predicted
0  0.593220   0.380588
1  0.067797   0.093926
2  0.313559   0.277727
3  0.186441   0.165619
4  0.067797   0.082072
5  0.177966   0.208313
6  0.466102   0.314531
7  0.127119   0.124035
8  0.254237   0.245801
9  0.271186   0.230941 

Feature Importances (Linear Regression Coefficients):
C6H6(GT)         0.416834
NOx(GT)          0.314221
PT08.S4(NO2)     0.209279
PT08.S1(CO)      0.150987
NMHC(GT)         0.099548
NO2(GT)          0.062535
PT08.S3(NOx)     0.037515
PT08.S2(NMHC)    0.016594
AH              -0.015117
RH              -0.048463
T               -0.091688
PT08.S5(O3)     -0.098886
dtype: float64
